<a href="https://colab.research.google.com/github/adelinewidyatmoko/ProjectA_Kelompok6_BanjirArticles_PBA/blob/main/notebooks/02_data_cleaning_banjir.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Cleaning & Analysis — Berita Banjir Indonesia

Notebook ini memproses hasil scraping berita banjir mulai dari cleaning, validasi keyword, sentiment analysis, hingga analisis perbandingan banjir biasa vs banjir bandang.

**Alur:**
1. Load CSV hasil scraping
2. Cleaning — hapus baris tanpa konten
3. Validasi relevansi keyword banjir
4. Sentiment Analysis (InSet Lexicon vs TextBlob)
5. Distribusi Banjir Biasa vs Banjir Bandang
6. Keyword Analysis — apa yang membedakan keduanya
7. Export dataset final

## 0. Install & Import

In [17]:
!pip install textblob -q
!pip install PySastrawi -q

In [18]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from textblob import TextBlob
from collections import Counter
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

print('Library siap!')

Library siap!


## 1. Load CSV Hasil Scraping

In [19]:
# upload file kalau di Colab
# from google.colab import files
# files.upload()  # upload data_berita_banjir_scraped.csv

df = pd.read_csv('data_berita_banjir_scraped.csv')

print(f'Total data loaded : {len(df)}')
print(f'Kolom             : {df.columns.tolist()}')
print(f'Konten kosong     : {df["Konten"].isna().sum()}')
print()
print('Distribusi kategori banjir (label manual):')
print(df['Sentimen'].value_counts())
df.head(3)

Total data loaded : 2454
Kolom             : ['No', 'Judul', 'Link', 'Sumber', 'Sentimen', 'Konten', 'Tanggal']
Konten kosong     : 800

Distribusi kategori banjir (label manual):
Sentimen
Banjir     1278
Bandang    1176
Name: count, dtype: int64


,No,Judul,Link,Sumber,Sentimen,Konten,Tanggal
0,1,Akses Jalan Perumnas Antang Lumpuh Akibat Banj...,https://www.detik.com/sulsel/makassar/d-837182...,Detik,Banjir,"Kawasan Perumnas Antang, Kota Makassar, Sulawe...",2026-02-25 11:43:23
1,2,"Banjir di Makassar, 545 Warga Pengungsi Dievak...",https://news.detik.com/berita/d-8371585/banjir...,Detik,Banjir,Badan Penanggulangan Bencana Daerah (BPBD) Mak...,2026-02-25 09:42:47
2,3,Motor Mogok di Jalan Inspeksi Kanal Borong Mak...,https://www.detik.com/sulsel/makassar/d-837154...,Detik,Banjir,"Waduk Tunggu Pampang, Kota Makassar, Sulawesi ...",2026-02-25 09:13:22


## 2. Cleaning — Hapus Baris Tanpa Konten

Artikel yang gagal di-scrape (konten kosong) tidak bisa dianalisis sehingga perlu dihapus dari dataset.

In [20]:
sebelum = len(df)

df = df[df['Konten'].notna()]
df = df[df['Konten'].astype(str).str.strip() != '']
df = df.reset_index(drop=True)

print(f'Sebelum cleaning  : {sebelum} artikel')
print(f'Sesudah cleaning  : {len(df)} artikel')
print(f'Dihapus           : {sebelum - len(df)} artikel')

Sebelum cleaning  : 2454 artikel
Sesudah cleaning  : 1654 artikel
Dihapus           : 800 artikel


## 3. Validasi Relevansi Keyword Banjir

Memastikan setiap artikel benar-benar membahas topik banjir. Artikel yang tidak mengandung kata kunci terkait banjir dihapus untuk menjaga kualitas dataset.

In [21]:
keywords_banjir = [
    'banjir', 'bandang', 'genangan', 'meluap', 'bencana',
    'terendam', 'mengungsi', 'pengungsi', 'banjir rob',
    'luapan', 'banjir kiriman', 'longsor'
]

pattern = '|'.join(keywords_banjir)
sebelum = len(df)
mask = df['Konten'].str.lower().str.contains(pattern, na=False)

df_tidak_relevan = df[~mask]
print(f'Artikel tidak relevan : {len(df_tidak_relevan)}')
if len(df_tidak_relevan) > 0:
    print('Contoh judul yang dihapus:')
    for judul in df_tidak_relevan['Judul'].head(5).tolist():
        print(f'  - {judul}')

df = df[mask].reset_index(drop=True)
print(f'\nDataset setelah filter : {len(df)} artikel')
print(f'Dihapus                : {sebelum - len(df)} artikel')

Artikel tidak relevan : 83
Contoh judul yang dihapus:
  - Jayapura Dilanda Banjir
  - Banjir di Nabire, Warga Sebut Belum Ada Saluran Pembuangan Air
  - Banjir & Longsor Di Kota Nabire Akibat Hujan Deras
  - Diguyur Hujan Deras Sejak Senin Sore Hingga Malam, Warga Smoker Siriwini Nabire Terendam Banjir
  - Banjir Mengisolasi Warga Di Distrik Makimi Nabire

Dataset setelah filter : 1571 artikel
Dihapus                : 83 artikel


## 4. Export Dataset Bersih

In [22]:
output = 'data_banjir_clean.csv'
df.to_csv(output, index=False)

print(f'Dataset tersimpan : {output}')
print(f'Total artikel     : {len(df)}')
print(f'Kolom             : {df.columns.tolist()}')
print()
print('Distribusi akhir:')
print(df['Sentimen'].value_counts())

# download (uncomment kalau di Colab)
# from google.colab import files
# files.download(output)

Dataset tersimpan : data_banjir_clean.csv
Total artikel     : 1571
Kolom             : ['No', 'Judul', 'Link', 'Sumber', 'Sentimen', 'Konten', 'Tanggal']

Distribusi akhir:
Sentimen
Bandang    909
Banjir     662
Name: count, dtype: int64
